# **Assignment 10: Deep Neural Networks**

## Author: Mohammad Mehdi Zare, Mohammad Ali Olama

#Load Libraries

In [ ]:
import pandas as pd
import io
import numpy as np
from sklearn.preprocessing import StandardScaler
import sklearn
import os
import copy
import seaborn as sns #visualisation
import matplotlib.pyplot as plt #visualisation
%matplotlib inline
sns.set(color_codes=True)

In [ ]:
# Diable warnings about max_itr not reached etc.!
def warn(*args, **kwargs):
    pass
import warnings
warnings.warn = warn

#Upload the Datasets
For this assignment, we used updated_pollution_dataset which we will use to do the classification task.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


##Classification dataset

In [ ]:
# os.environ['KAGGLE_CONFIG_DIR'] = "/content/gdrive/My Drive/Kaggle"
# ! kaggle datasets download -d mujtabamatin/air-quality-and-pollution-assessment
# !unzip \*.zip && rm *.zip
df = pd.read_csv('updated_pollution_dataset.csv')
print(df.shape)

(5000, 10)


In [ ]:
df.head()

,Temperature,Humidity,PM2.5,PM10,NO2,SO2,CO,Proximity_to_Industrial_Areas,Population_Density,Air Quality
0,29.8,59.1,5.2,17.9,18.9,9.2,1.72,6.3,319,Moderate
1,28.3,75.6,2.3,12.2,30.8,9.7,1.64,6.0,611,Moderate
2,23.1,74.7,26.7,33.8,24.4,12.6,1.63,5.2,619,Moderate
3,27.1,39.1,6.1,6.3,13.5,5.3,1.15,11.1,551,Good
4,26.5,70.7,6.9,16.0,21.9,5.6,1.01,12.7,303,Good


##Preprocessing

In [ ]:
null_columns = []
for col_name in list(df.columns):
  no_null_values = sum(df[col_name].isnull())
  if no_null_values:
    print('Number of null values in column %s is %d' %(col_name,no_null_values))
    null_columns.append(col_name)

In [ ]:
df.dropna(inplace=True)
df.shape

(5000, 10)

In [ ]:
target_variable = df.pop('Air Quality')

In [ ]:
from sklearn.preprocessing import OneHotEncoder,LabelBinarizer
enc = LabelBinarizer()
y = enc.fit_transform(target_variable)

#Split dataset

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score

X_train, X_test, y_train, y_test = train_test_split(df,y,test_size=0.2,random_state=42,stratify=y)

print(X_train.shape)
print(X_test.shape)

(4000, 9)
(1000, 9)


#Classification

##Keras

In [ ]:
import pandas as pd
import io
import numpy as np
from sklearn.preprocessing import StandardScaler
import sklearn
import os
import copy
import seaborn as sns #visualisation
import matplotlib.pyplot as plt #visualisation
import warnings
from google.colab import drive
from sklearn.preprocessing import OneHotEncoder,LabelBinarizer
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import classification_report
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense , Dropout
from sklearn.model_selection import KFold


keras.utils.set_random_seed(42)

In [ ]:
def create_model(input_dim, num_classes, hidden_layers, hidden_units, optimizer = 'adam',  activation='relu' , regularizer=None , activity_regularizer=None, dropout_rate=0.0):
    model = Sequential()
    model.add(Dense(hidden_units[0], activation=activation, input_dim=input_dim , kernel_regularizer=regularizer , activity_regularizer=activity_regularizer ))
    model.add(Dropout(dropout_rate))
    for units in hidden_units[1:]:
        model.add(Dense(units, activation=activation, kernel_regularizer=regularizer, activity_regularizer=activity_regularizer))
        model.add(Dropout(dropout_rate))
    model.add(Dense(num_classes, activation='softmax'))  # Sigmoid for binary classification
    model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])
    return model

## Tuning optimization algorithm

In [ ]:
from tensorflow.keras.optimizers import SGD, Adam, RMSprop, Adadelta, Adagrad

epoch = 20
batch_size = 256
optimizers = ['adam', 'sgd', 'rmsprop', 'adagrad', 'adadelta']

params= {'hidden_layers': 4, 'hidden_units': [64, 30 , 15 , 8]}

results = {}

# Loop through each optimizer
for optimizer_name in optimizers:
    print(f"\nTraining with optimizer: {optimizer_name}")
    print(f"Training with parameters: {params}")

    # Initialize KFold
    kf = KFold(n_splits=4, shuffle=True, random_state=42)

    # Store the results of each fold
    accuracy_scores = []

    # Iterate over the folds
    for train_index, val_index in kf.split(X_train):
        X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
        y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]

        # Create and train the model
        model = create_model(input_dim=X_train.shape[1], num_classes=4,
                          hidden_layers=params['hidden_layers'], hidden_units=params['hidden_units'],
                          optimizer=optimizer_name)

        model.fit(X_fold_train, y_fold_train, epochs=epoch, batch_size=batch_size, verbose=0) # Adjust epochs and batch size as needed

        # Evaluate the model
        _, accuracy = model.evaluate(X_fold_val, y_fold_val, verbose=0)
        accuracy_scores.append(accuracy)

    # Print the results
    print("Accuracy scores for each fold:", accuracy_scores)
    print("Average accuracy:", np.mean(accuracy_scores))
    results.update({optimizer_name : np.mean(accuracy_scores)})
    print(40 * '*')

print(results)


Training with optimizer: adam
Training with parameters: {'hidden_layers': 4, 'hidden_units': [64, 30, 15, 8]}
Accuracy scores for each fold: [0.6930000185966492, 0.32899999618530273, 0.48500001430511475, 0.703000009059906]
Average accuracy: 0.5525000095367432
****************************************

Training with optimizer: sgd
Training with parameters: {'hidden_layers': 4, 'hidden_units': [64, 30, 15, 8]}
Accuracy scores for each fold: [0.4580000042915344, 0.421999990940094, 0.38600000739097595, 0.4059999883174896]
Average accuracy: 0.4179999977350235
****************************************

Training with optimizer: rmsprop
Training with parameters: {'hidden_layers': 4, 'hidden_units': [64, 30, 15, 8]}
Accuracy scores for each fold: [0.38600000739097595, 0.43700000643730164, 0.3140000104904175, 0.4189999997615814]
Average accuracy: 0.3890000060200691
****************************************

Training with optimizer: adagrad
Training with parameters: {'hidden_layers': 4, 'hidden_uni

In [ ]:
import operator

best_optimizer = max(results.items(), key=operator.itemgetter(1))
print(f"The best optimizer is {best_optimizer[0]} with an accuracy of {best_optimizer[1]}")


model = create_model(input_dim=X_train.shape[1], num_classes=4,  # Assuming binary classification
                             hidden_layers=params['hidden_layers'], hidden_units=params['hidden_units'],
                             optimizer=best_optimizer[0])

model.fit(X_train, y_train, epochs=100, batch_size=batch_size) # Adjust epochs and batch size as needed

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Make predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis = 1)

# Print classification report
print(classification_report(y_true_classes, y_pred_classes))


The best optimizer is adam with an accuracy of 0.5525000095367432
Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.1371 - loss: 6.1992 
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1961 - loss: 1.3697 
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.2451 - loss: 0.6823 
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.3870 - loss: 0.5830  
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4436 - loss: 0.5623 
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4580 - loss: 0.5455 
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4747 - loss: 0.5349 
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4935 - loss: 0.5260 
Epoch 9/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5096 - loss: 0.5186  
Epoch 10/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5222 - loss: 0.5124 
Epoch 11/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5388 

## Tune learning rate


In [ ]:
from sklearn.model_selection import KFold
learning_rates = [0.003,  0.01, 0.03 , 0.1 , 0.3]
results_lr = {}

for lr in learning_rates:
    print(f"\nTraining with learning rate: {lr}")

    # Initialize KFold
    kf = KFold(n_splits=4, shuffle=True, random_state=42)

    accuracy_scores = []
    for train_index, val_index in kf.split(X_train):
        X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
        y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]
        optimizer = Adam(learning_rate=lr)

        model = create_model(input_dim=X_train.shape[1], num_classes=4,
                             hidden_layers=params['hidden_layers'], hidden_units=params['hidden_units'],
                             optimizer=optimizer)
        model.fit(X_fold_train, y_fold_train, epochs=epoch, batch_size=batch_size, verbose=0)

        _, accuracy = model.evaluate(X_fold_val, y_fold_val, verbose=0)
        accuracy_scores.append(accuracy)

    print("Accuracy scores for each fold:", accuracy_scores)
    print("Average accuracy:", np.mean(accuracy_scores))
    results_lr.update({lr : np.mean(accuracy_scores)})

print(results_lr)
best_lr = max(results_lr.items(), key=operator.itemgetter(1))
print(f"The best learning rate is {best_lr[0]} with an accuracy of {best_lr[1]}")


Training with learning rate: 0.003
Accuracy scores for each fold: [0.7960000038146973, 0.7770000100135803, 0.6399999856948853, 0.4059999883174896]
Average accuracy: 0.6547499969601631

Training with learning rate: 0.01
Accuracy scores for each fold: [0.6650000214576721, 0.421999990940094, 0.38600000739097595, 0.4059999883174896]
Average accuracy: 0.4697500020265579

Training with learning rate: 0.03
Accuracy scores for each fold: [0.38600000739097595, 0.8370000123977661, 0.38600000739097595, 0.4059999883174896]
Average accuracy: 0.5037500038743019

Training with learning rate: 0.1
Accuracy scores for each fold: [0.38600000739097595, 0.421999990940094, 0.38600000739097595, 0.4059999883174896]
Average accuracy: 0.3999999985098839

Training with learning rate: 0.3
Accuracy scores for each fold: [0.38600000739097595, 0.421999990940094, 0.38600000739097595, 0.4059999883174896]
Average accuracy: 0.3999999985098839
{0.003: 0.6547499969601631, 0.01: 0.4697500020265579, 0.03: 0.503750003874301

In [ ]:
import operator

optimizer = Adam(learning_rate=best_lr[0])


model = create_model(input_dim=X_train.shape[1], num_classes=4,
                             hidden_layers=params['hidden_layers'], hidden_units=params['hidden_units'],
                             optimizer=optimizer)

model.fit(X_train, y_train, epochs=100, batch_size=batch_size)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Make predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis = 1)

print(classification_report(y_true_classes, y_pred_classes))


Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.3062 - loss: 4.4007
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2042 - loss: 0.6117
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.2013 - loss: 0.5957
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1999 - loss: 0.5843
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1999 - loss: 0.5751 
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1998 - loss: 0.5685  
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.1995 - loss: 0.5617 
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1989 - loss: 0.5565
Epoch 9/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2173 - loss: 0.5516 
Epoch 10/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2893 - loss: 0.5474 
Epoch 11/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.3878 - loss: 0.5439 
Epoch 12/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - ac

## Tune learning rate decay

In [ ]:
learning_rate_decays = [0.001, 0.01,0.03, 0.1, 0.3]
results_lr_decay = {}

for decay in learning_rate_decays:
    print(f"\nTraining with learning rate decay: {decay}")

    # Initialize KFold
    kf = KFold(n_splits=4, shuffle=True, random_state=42)

    accuracy_scores = []
    for train_index, val_index in kf.split(X_train):
        X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
        y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]
        optimizer = Adam(learning_rate=best_lr[0], decay=decay) # Apply decay here

        model = create_model(input_dim=X_train.shape[1], num_classes=4,
                             hidden_layers=params['hidden_layers'], hidden_units=params['hidden_units'],
                             optimizer=optimizer)
        model.fit(X_fold_train, y_fold_train, epochs=epoch, batch_size=batch_size, verbose=0)

        _, accuracy = model.evaluate(X_fold_val, y_fold_val, verbose=0)
        accuracy_scores.append(accuracy)

    print("Accuracy scores for each fold:", accuracy_scores)
    print("Average accuracy:", np.mean(accuracy_scores))
    results_lr_decay.update({decay : np.mean(accuracy_scores)})

print(results_lr_decay)
best_decay = max(results_lr_decay.items(), key=operator.itemgetter(1))
print(f"The best learning rate decay is {best_decay[0]} with an accuracy of {best_decay[1]}")



Training with learning rate decay: 0.001
Accuracy scores for each fold: [0.38600000739097595, 0.421999990940094, 0.38600000739097595, 0.6240000128746033]
Average accuracy: 0.4545000046491623

Training with learning rate decay: 0.01
Accuracy scores for each fold: [0.6890000104904175, 0.7129999995231628, 0.6430000066757202, 0.4059999883174896]
Average accuracy: 0.6127500012516975

Training with learning rate decay: 0.03
Accuracy scores for each fold: [0.6480000019073486, 0.421999990940094, 0.7839999794960022, 0.4059999883174896]
Average accuracy: 0.5649999901652336

Training with learning rate decay: 0.1
Accuracy scores for each fold: [0.6549999713897705, 0.527999997138977, 0.6850000023841858, 0.6549999713897705]
Average accuracy: 0.630749985575676

Training with learning rate decay: 0.3
Accuracy scores for each fold: [0.6349999904632568, 0.4620000123977661, 0.38600000739097595, 0.6620000004768372]
Average accuracy: 0.536250002682209
{0.001: 0.4545000046491623, 0.01: 0.6127500012516975,

In [ ]:
optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])
print(optimizer.learning_rate)
model = create_model(input_dim=X_train.shape[1], num_classes=4,
                             hidden_layers=params['hidden_layers'], hidden_units=params['hidden_units'],
                             optimizer=optimizer)

model.fit(X_train, y_train, epochs=100, batch_size=batch_size)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Make predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis = 1)

print(classification_report(y_true_classes, y_pred_classes))

<KerasVariable shape=(), dtype=float32, path=adam/learning_rate>
Epoch 1/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.3014 - loss: 1.9134
Epoch 2/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4053 - loss: 0.6262  
Epoch 3/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4679 - loss: 0.5942 
Epoch 4/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4595 - loss: 0.5493 
Epoch 5/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4728 - loss: 0.5224  
Epoch 6/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4984 - loss: 0.5006 
Epoch 7/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5120 - loss: 0.4765 
Epoch 8/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5125 - loss: 0.4536  
Epoch 9/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5290 - loss: 0.4504  
Epoch 10/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5320 - loss: 0.4418 
Epoch 11/100
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.5372 

In [ ]:
batch_sizes = [16, 32, 64, 128, 256]
results_batch = {}

for batch_size in batch_sizes:
    print(f"\nTraining with batch size: {batch_size}")
    # Using the best optimizer, learning rate, and decay from previous sections

    # Initialize KFold
    kf = KFold(n_splits=4, shuffle=True, random_state=42)

    accuracy_scores = []
    for train_index, val_index in kf.split(X_train):
        X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
        y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]
        optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])

        model = create_model(input_dim=X_train.shape[1], num_classes=4,
                             hidden_layers=params['hidden_layers'], hidden_units=params['hidden_units'],
                             optimizer=optimizer)
        model.fit(X_fold_train, y_fold_train, epochs=epoch, batch_size=batch_size, verbose=0)

        _, accuracy = model.evaluate(X_fold_val, y_fold_val, verbose=0)
        accuracy_scores.append(accuracy)

    print("Accuracy scores for each fold:", accuracy_scores)
    print("Average accuracy:", np.mean(accuracy_scores))
    results_batch.update({batch_size: np.mean(accuracy_scores)})

print(results_batch)
best_batch_size = max(results_batch.items(), key=operator.itemgetter(1))
print(f"The best batch size is {best_batch_size[0]} with an accuracy of {best_batch_size[1]}")



Training with batch size: 16
Accuracy scores for each fold: [0.8159999847412109, 0.7390000224113464, 0.8489999771118164, 0.4059999883174896]
Average accuracy: 0.7024999931454659

Training with batch size: 32
Accuracy scores for each fold: [0.7540000081062317, 0.8510000109672546, 0.8209999799728394, 0.8320000171661377]
Average accuracy: 0.8145000040531158

Training with batch size: 64
Accuracy scores for each fold: [0.7689999938011169, 0.4230000078678131, 0.8180000185966492, 0.8399999737739563]
Average accuracy: 0.7124999985098839

Training with batch size: 128
Accuracy scores for each fold: [0.38600000739097595, 0.421999990940094, 0.38600000739097595, 0.4059999883174896]
Average accuracy: 0.3999999985098839

Training with batch size: 256
Accuracy scores for each fold: [0.38600000739097595, 0.421999990940094, 0.38600000739097595, 0.7009999752044678]
Average accuracy: 0.4737499952316284
{16: 0.7024999931454659, 32: 0.8145000040531158, 64: 0.7124999985098839, 128: 0.3999999985098839, 256

In [ ]:
# Now train the model with the best hyperparameters found so far, including the best batch size
optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])
model = create_model(input_dim=X_train.shape[1], num_classes=4,
                     hidden_layers=params['hidden_layers'], hidden_units=params['hidden_units'],
                     optimizer=optimizer)

model.fit(X_train, y_train, epochs=100, batch_size=128)

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Make predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(classification_report(y_true_classes, y_pred_classes))

Epoch 1/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.2632 - loss: 7.9813
Epoch 2/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4026 - loss: 0.6733
Epoch 3/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4026 - loss: 0.6562
Epoch 4/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4026 - loss: 0.6405
Epoch 5/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4026 - loss: 0.6265
Epoch 6/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4026 - loss: 0.6140
Epoch 7/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4026 - loss: 0.6031
Epoch 8/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4026 - loss: 0.5936
Epoch 9/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4026 - loss: 0.5853
Epoch 10/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4026 - loss: 0.5781
Epoch 11/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.4026 - loss: 0.5718
Epoch 12/100
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy:

## tune activation function

In [ ]:
activations = ['relu', 'sigmoid', 'tanh', 'elu', 'selu']
results_activation = {}

for activation_func in activations:
    print(f"\nTraining with activation function: {activation_func}")
    # Using the best optimizer, learning rate, decay, and batch size

    # Initialize KFold
    kf = KFold(n_splits=4, shuffle=True, random_state=42)

    accuracy_scores = []
    for train_index, val_index in kf.split(X_train):
        X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
        y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]
        optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])

        model = create_model(input_dim=X_train.shape[1], num_classes=4,
                             hidden_layers=params['hidden_layers'], hidden_units=params['hidden_units'],
                             optimizer=optimizer, activation=activation_func)
        model.fit(X_fold_train, y_fold_train, epochs=epoch, batch_size=best_batch_size[0], verbose=0)

        _, accuracy = model.evaluate(X_fold_val, y_fold_val, verbose=0)
        accuracy_scores.append(accuracy)

    print("Accuracy scores for each fold:", accuracy_scores)
    print("Average accuracy:", np.mean(accuracy_scores))
    results_activation.update({activation_func: np.mean(accuracy_scores)})

print(results_activation)
best_activation = max(results_activation.items(), key=operator.itemgetter(1))
print(f"The best activation function is {best_activation[0]} with an accuracy of {best_activation[1]}")




Training with activation function: relu
Accuracy scores for each fold: [0.8349999785423279, 0.421999990940094, 0.38600000739097595, 0.8500000238418579]
Average accuracy: 0.6232500001788139

Training with activation function: sigmoid
Accuracy scores for each fold: [0.8019999861717224, 0.8299999833106995, 0.8050000071525574, 0.8140000104904175]
Average accuracy: 0.8127499967813492

Training with activation function: tanh
Accuracy scores for each fold: [0.7260000109672546, 0.6899999976158142, 0.6869999766349792, 0.6579999923706055]
Average accuracy: 0.6902499943971634

Training with activation function: elu
Accuracy scores for each fold: [0.38600000739097595, 0.9229999780654907, 0.824999988079071, 0.8820000290870667]
Average accuracy: 0.7540000006556511

Training with activation function: selu
Accuracy scores for each fold: [0.8659999966621399, 0.925000011920929, 0.38600000739097595, 0.8899999856948853]
Average accuracy: 0.7667500004172325
{'relu': 0.6232500001788139, 'sigmoid': 0.812749

In [ ]:
#Now train the model with all best hyperparameters
optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])
model = create_model(input_dim=X_train.shape[1], num_classes=4,
                     hidden_layers=params['hidden_layers'], hidden_units=params['hidden_units'],
                     optimizer=optimizer, activation=best_activation[0])

model.fit(X_train, y_train, epochs=100, batch_size=best_batch_size[0])

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Make predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(classification_report(y_true_classes, y_pred_classes))

Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 3s 4ms/step - accuracy: 0.1617 - loss: 0.6745
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.4012 - loss: 0.5318
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4723 - loss: 0.5028
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.6599 - loss: 0.3740
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6733 - loss: 0.3298
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6831 - loss: 0.3027
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7535 - loss: 0.2846
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7766 - loss: 0.2679
Epoch 9/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7807 - loss: 0.2521
Epoch 10/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7920 - loss: 0.2375
Epoch 11/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.7986 - loss: 0.2279
Epoch 12/100
125/125 ━━━━━━━━━━━━━━━━━━━━

## tune weight initialization


In [ ]:
weight_initializers = ['random_normal', 'random_uniform', 'glorot_uniform', 'glorot_normal', 'he_normal', 'he_uniform']
results_weight_init = {}

for weight_init in weight_initializers:
    print(f"\nTraining with weight initializer: {weight_init}")
    # Using the best optimizer, learning rate, decay, batch size, and activation function

    # Initialize KFold
    kf = KFold(n_splits=4, shuffle=True, random_state=42)

    accuracy_scores = []
    for train_index, val_index in kf.split(X_train):
        X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
        y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]
        optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])

        model = Sequential()
        model.add(Dense(params['hidden_units'][0], activation=best_activation[0], input_dim=X_train.shape[1], kernel_initializer=weight_init))
        for units in params['hidden_units'][1:]:
            model.add(Dense(units, activation=best_activation[0], kernel_initializer=weight_init))
        model.add(Dense(4, activation='softmax')) # Output layer
        model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

        model.fit(X_fold_train, y_fold_train, epochs=epoch, batch_size=best_batch_size[0], verbose=0)

        _, accuracy = model.evaluate(X_fold_val, y_fold_val, verbose=0)
        accuracy_scores.append(accuracy)

    print("Accuracy scores for each fold:", accuracy_scores)
    print("Average accuracy:", np.mean(accuracy_scores))
    results_weight_init.update({weight_init: np.mean(accuracy_scores)})

print(results_weight_init)
best_weight_init = max(results_weight_init.items(), key=operator.itemgetter(1))
print(f"The best weight initializer is {best_weight_init[0]} with an accuracy of {best_weight_init[1]}")



Training with weight initializer: random_normal
Accuracy scores for each fold: [0.6800000071525574, 0.421999990940094, 0.38600000739097595, 0.6359999775886536]
Average accuracy: 0.5309999957680702

Training with weight initializer: random_uniform
Accuracy scores for each fold: [0.38600000739097595, 0.421999990940094, 0.6880000233650208, 0.4059999883174896]
Average accuracy: 0.4755000025033951

Training with weight initializer: glorot_uniform
Accuracy scores for each fold: [0.6790000200271606, 0.6990000009536743, 0.7609999775886536, 0.8109999895095825]
Average accuracy: 0.7374999970197678

Training with weight initializer: glorot_normal
Accuracy scores for each fold: [0.7990000247955322, 0.7789999842643738, 0.7879999876022339, 0.7490000128746033]
Average accuracy: 0.7787500023841858

Training with weight initializer: he_normal
Accuracy scores for each fold: [0.7549999952316284, 0.7990000247955322, 0.746999979019165, 0.7609999775886536]
Average accuracy: 0.7654999941587448

Training wit

In [ ]:
# Now train the model with all best hyperparameters including weight initializer
optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])
model = Sequential()
model.add(Dense(params['hidden_units'][0], activation=best_activation[0], input_dim=X_train.shape[1], kernel_initializer=best_weight_init[0]))
for units in params['hidden_units'][1:]:
    model.add(Dense(units, activation=best_activation[0], kernel_initializer=best_weight_init[0]))
model.add(Dense(4, activation='softmax'))
model.compile(loss='binary_crossentropy', optimizer=optimizer, metrics=['accuracy'])

model.fit(X_train, y_train, epochs=100, batch_size=best_batch_size[0])

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Make predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(classification_report(y_true_classes, y_pred_classes))

Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.4012 - loss: 0.5872
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.4012 - loss: 0.5250
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.6076 - loss: 0.4231
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6606 - loss: 0.3408
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6745 - loss: 0.3144
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6783 - loss: 0.2999
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.6778 - loss: 0.2848
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7660 - loss: 0.2684
Epoch 9/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7583 - loss: 0.2541
Epoch 10/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7645 - loss: 0.2509
Epoch 11/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7759 - loss: 0.2399
Epoch 12/100
125/125 ━━━━━━━━━━━━━━━━━━━━

## Fine tune architucture

In [ ]:
# Define a list of architectures to try
architectures = {
    '3_layers': [32, 16, 8],
    '5_layers': [34, 28, 20, 12, 8],
    '6_layers': [28, 24, 20, 16, 12, 8],
    '7_layers': [32, 28, 24, 20, 16, 12, 8],
    '10_layers': [32, 30, 28, 24, 20, 20, 16, 12, 12, 8]
}

results_architecture = {}

for arch_name, hidden_units in architectures.items():
    print(f"\nTraining with architecture: {arch_name}")
    # Using the best hyperparameters from previous sections

    # Initialize KFold
    kf = KFold(n_splits=4, shuffle=True, random_state=42)

    accuracy_scores = []
    for train_index, val_index in kf.split(X_train):
        X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
        y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]
        optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])


        model = create_model(input_dim=X_train.shape[1], num_classes=4,
                             hidden_layers=arch_name, hidden_units=hidden_units,
                             optimizer=optimizer)
        model.fit(X_fold_train, y_fold_train, epochs=epoch, batch_size=batch_size, verbose=0)

        _, accuracy = model.evaluate(X_fold_val, y_fold_val, verbose=0)
        accuracy_scores.append(accuracy)

    print("Accuracy scores for each fold:", accuracy_scores)
    print("Average accuracy:", np.mean(accuracy_scores))
    results_architecture.update({arch_name: np.mean(accuracy_scores)})

print(results_architecture)
best_architecture = max(results_architecture.items(), key=operator.itemgetter(1))
print(f"The best architecture is {best_architecture[0]} with an accuracy of {best_architecture[1]}")


Training with architecture: 3_layers
Accuracy scores for each fold: [0.38600000739097595, 0.421999990940094, 0.38600000739097595, 0.4059999883174896]
Average accuracy: 0.3999999985098839

Training with architecture: 5_layers
Accuracy scores for each fold: [0.39899998903274536, 0.7260000109672546, 0.4779999852180481, 0.6620000004768372]
Average accuracy: 0.5662499964237213

Training with architecture: 6_layers
Accuracy scores for each fold: [0.703000009059906, 0.7919999957084656, 0.460999995470047, 0.7990000247955322]
Average accuracy: 0.6887500062584877

Training with architecture: 7_layers
Accuracy scores for each fold: [0.8040000200271606, 0.7990000247955322, 0.777999997138977, 0.6570000052452087]
Average accuracy: 0.7595000118017197

Training with architecture: 10_layers
Accuracy scores for each fold: [0.8199999928474426, 0.6959999799728394, 0.8389999866485596, 0.7570000290870667]
Average accuracy: 0.777999997138977
{'3_layers': 0.3999999985098839, '5_layers': 0.5662499964237213, '

In [ ]:
# Now train the model with the best architecture and all other best hyperparameters
best_hidden_units = architectures[best_architecture[0]]
optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])
model = create_model(input_dim=X_train.shape[1], num_classes=4,
                      hidden_layers=arch_name, hidden_units=best_hidden_units,
                      optimizer=optimizer)
model.fit(X_train, y_train, epochs=100, batch_size=best_batch_size[0])

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Make predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(classification_report(y_true_classes, y_pred_classes))

Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 4s 3ms/step - accuracy: 0.4457 - loss: 0.5494
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6062 - loss: 0.3894
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7291 - loss: 0.2869
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7386 - loss: 0.2622
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7872 - loss: 0.2291
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7927 - loss: 0.2171
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8147 - loss: 0.2005
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8109 - loss: 0.2018
Epoch 9/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8417 - loss: 0.1877
Epoch 10/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.8411 - loss: 0.1878
Epoch 11/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8288 - loss: 0.1938
Epoch 12/100
125/125 ━━━━━━━━━━━━━━━━━━━━

## Tuning l1 and l2 regularization in the weights and activity_kernel (together)

In [ ]:
from tensorflow.keras.regularizers import l1, l2, l1_l2

l1_values = [0.0000001, 0.00001]
l2_values = [0.0000001, 0.00001]

results_regularization = {}

for l1_val in l1_values:
    for l2_val in l2_values:
        print(f"\nTraining with L1 regularization: {l1_val}, L2 regularization: {l2_val}")
        # Using the best hyperparameters from previous sections

        # Initialize KFold
        kf = KFold(n_splits=4, shuffle=True, random_state=42)

        accuracy_scores = []
        for train_index, val_index in kf.split(X_train):
            X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
            y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]
            optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])
            regularizer = l1_l2(l1=l1_val, l2=l2_val)
            activity_regularizer = l1_l2(l1=l1_val, l2=l2_val)

            model = create_model(input_dim=X_train.shape[1], num_classes=4,
                                  hidden_layers=arch_name, hidden_units=best_hidden_units,
                                  optimizer=optimizer, regularizer=regularizer , activity_regularizer=activity_regularizer)
            model.fit(X_fold_train, y_fold_train, epochs=epoch, batch_size=best_batch_size[0], verbose=0)

            _, accuracy = model.evaluate(X_fold_val, y_fold_val, verbose=0)
            accuracy_scores.append(accuracy)

        print("Accuracy scores for each fold:", accuracy_scores)
        print("Average accuracy:", np.mean(accuracy_scores))
        results_regularization.update({(l1_val, l2_val): np.mean(accuracy_scores)})

print(results_regularization)
best_regularization = max(results_regularization.items(), key=operator.itemgetter(1))
print(f"The best L1 and L2 regularization values are {best_regularization[0]} with an accuracy of {best_regularization[1]}")




Training with L1 regularization: 1e-07, L2 regularization: 1e-07
Accuracy scores for each fold: [0.9070000052452087, 0.9120000004768372, 0.8859999775886536, 0.8999999761581421]
Average accuracy: 0.9012499898672104

Training with L1 regularization: 1e-07, L2 regularization: 1e-05
Accuracy scores for each fold: [0.8949999809265137, 0.9049999713897705, 0.8949999809265137, 0.9039999842643738]
Average accuracy: 0.8997499793767929

Training with L1 regularization: 1e-05, L2 regularization: 1e-07
Accuracy scores for each fold: [0.9100000262260437, 0.8880000114440918, 0.875, 0.9049999713897705]
Average accuracy: 0.8945000022649765

Training with L1 regularization: 1e-05, L2 regularization: 1e-05
Accuracy scores for each fold: [0.8640000224113464, 0.6940000057220459, 0.8510000109672546, 0.8920000195503235]
Average accuracy: 0.8252500146627426
{(1e-07, 1e-07): 0.9012499898672104, (1e-07, 1e-05): 0.8997499793767929, (1e-05, 1e-07): 0.8945000022649765, (1e-05, 1e-05): 0.8252500146627426}
The best

In [ ]:
# Now train the model with the best hyperparameters and regularization
best_l1, best_l2 = best_regularization[0]
optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])
regularizer = l1_l2(l1=best_l1, l2=best_l2)
activity_regularizer = l1_l2(l1=best_l1, l2=best_l2)

model = create_model(input_dim=X_train.shape[1], num_classes=4,
                      hidden_layers=arch_name, hidden_units=best_hidden_units,
                      optimizer=optimizer, regularizer=regularizer , activity_regularizer=activity_regularizer)

model.fit(X_train, y_train, epochs=100, batch_size=best_batch_size[0])

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Make predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(classification_report(y_true_classes, y_pred_classes))

Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.4884 - loss: 1.1724
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7221 - loss: 0.4027
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7713 - loss: 0.2965
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7687 - loss: 0.2838
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.7955 - loss: 0.2436
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8328 - loss: 0.2155
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8332 - loss: 0.2122
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8522 - loss: 0.1890
Epoch 9/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8540 - loss: 0.1853
Epoch 10/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8723 - loss: 0.1723
Epoch 11/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.8654 - loss: 0.1770
Epoch 12/100
125/125 ━━━━━━━━━━━━━━━━━━━━

## Tuning dropout rate

In [ ]:
dropout_rates = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]  # Example dropout rates to try
results_dropout = {}

for dropout_rate in dropout_rates:
    print(f"\nTraining with dropout rate: {dropout_rate}")
    # Using the best hyperparameters from previous sections

    # Initialize KFold
    kf = KFold(n_splits=4, shuffle=True, random_state=42)

    accuracy_scores = []
    for train_index, val_index in kf.split(X_train):
        X_fold_train, X_fold_val = X_train.iloc[train_index], X_train.iloc[val_index]
        y_fold_train, y_fold_val = y_train[train_index], y_train[val_index]
        optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])
        regularizer = l1_l2(l1=best_l1, l2=best_l2)
        activity_regularizer = l1_l2(l1=best_l1, l2=best_l2)

        model = create_model(input_dim=X_train.shape[1], num_classes=4,
                              hidden_layers=arch_name, hidden_units=best_hidden_units,
                              optimizer=optimizer, regularizer=regularizer,
                              activity_regularizer=activity_regularizer,
                              dropout_rate=dropout_rate) # Pass dropout rate to model creation
        model.fit(X_fold_train, y_fold_train, epochs=epoch, batch_size=best_batch_size[0], verbose=0)

        _, accuracy = model.evaluate(X_fold_val, y_fold_val, verbose=0)
        accuracy_scores.append(accuracy)

    print("Accuracy scores for each fold:", accuracy_scores)
    print("Average accuracy:", np.mean(accuracy_scores))
    results_dropout.update({dropout_rate: np.mean(accuracy_scores)})

print(results_dropout)
best_dropout_rate = max(results_dropout.items(), key=operator.itemgetter(1))
print(f"The best dropout rate is {best_dropout_rate[0]} with an accuracy of {best_dropout_rate[1]}")





Training with dropout rate: 0.0
Accuracy scores for each fold: [0.8989999890327454, 0.9110000133514404, 0.8949999809265137, 0.9039999842643738]
Average accuracy: 0.9022499918937683

Training with dropout rate: 0.1
Accuracy scores for each fold: [0.8140000104904175, 0.8650000095367432, 0.8659999966621399, 0.6930000185966492]
Average accuracy: 0.8095000088214874

Training with dropout rate: 0.2
Accuracy scores for each fold: [0.6809999942779541, 0.5989999771118164, 0.6919999718666077, 0.7260000109672546]
Average accuracy: 0.6744999885559082

Training with dropout rate: 0.3
Accuracy scores for each fold: [0.38600000739097595, 0.6899999976158142, 0.6019999980926514, 0.4059999883174896]
Average accuracy: 0.5209999978542328

Training with dropout rate: 0.4
Accuracy scores for each fold: [0.38600000739097595, 0.421999990940094, 0.38600000739097595, 0.4059999883174896]
Average accuracy: 0.3999999985098839

Training with dropout rate: 0.5
Accuracy scores for each fold: [0.38600000739097595, 0.

In [ ]:
# Now train the model with the best hyperparameters and dropout rate
optimizer = Adam(learning_rate=best_lr[0], decay=best_decay[0])
regularizer = l1_l2(l1=best_l1, l2=best_l2)
activity_regularizer = l1_l2(l1=best_l1, l2=best_l2)

model = create_model(input_dim=X_train.shape[1], num_classes=4,
                      hidden_layers=arch_name, hidden_units=best_hidden_units,
                      optimizer=optimizer, regularizer=regularizer,
                      activity_regularizer=activity_regularizer,
                      dropout_rate=best_dropout_rate[0]) # Use best dropout rate

model.fit(X_train, y_train, epochs=100, batch_size=best_batch_size[0])

# Evaluate the model
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {loss}")
print(f"Test Accuracy: {accuracy}")

# Make predictions
y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)
y_true_classes = np.argmax(y_test, axis=1)

print(classification_report(y_true_classes, y_pred_classes))

Epoch 1/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - accuracy: 0.3121 - loss: 1.0194
Epoch 2/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.6644 - loss: 0.3715
Epoch 3/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.7499 - loss: 0.2807
Epoch 4/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.7998 - loss: 0.2271
Epoch 5/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8399 - loss: 0.2037
Epoch 6/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8521 - loss: 0.1839
Epoch 7/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8667 - loss: 0.1716
Epoch 8/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8705 - loss: 0.1642
Epoch 9/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8757 - loss: 0.1579
Epoch 10/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8826 - loss: 0.1537
Epoch 11/100
125/125 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.8820 - loss: 0.1518
Epoch 12/100
125/125 ━━━━━━━━━━━━━━━━━━━━

# In a paragraph, explain why it gets more difficult to train deep neural networks when the number of layers increase (i.e. when the network gets deeper).

Training deep neural networks becomes increasingly difficult as the number of layers grows due to several challenges. One major issue is the **vanishing or exploding gradient** problem, where gradients diminish or grow uncontrollably during backpropagation, making it harder for earlier layers to learn effectively. Deeper networks also encounter **optimization difficulties**, as their loss landscapes are more complex, with many local minima and saddle points that hinder convergence. Additionally, the **risk of overfitting** rises, especially if the training data is insufficient, necessitating careful regularization and more extensive datasets. Deeper architectures also demand **more computational power** and time, further complicating training. Techniques like residual connections, batch normalization, and gradient clipping are often required to mitigate these issues and enable successful training of deep networks.